# Setup

In [ ]:
# load required libraries with requirements.txt
! pip install -r requirements.txt

In [52]:
import matplotlib.pyplot as plt
import mysql.connector as mysql
import pandas as pd
import numpy as np
import json

from DbConnector import DbConnector

In [ ]:
# Import datasets
credits_df = pd.read_csv("movies_ds/credits.csv")
keywords_df = pd.read_csv("movies_ds/keywords.csv")
links_small_df = pd.read_csv("movies_ds/links_small.csv")
links_df = pd.read_csv("movies_ds/links.csv")
movies_metadata_df = pd.read_csv("movies_ds/movies_metadata.csv")
ratings_small_df = pd.read_csv("movies_ds/ratings_small.csv")
ratings_df = pd.read_csv("movies_ds/ratings.csv")

/var/folders/vl/xlcg7z052993rf2f_9vpl4h00000gn/T/ipykernel_51461/1606664570.py:6: DtypeWarning: Columns (10) have mixed types. Specify dtype option on import or set low_memory=False.
  movies_metadata_df = pd.read_csv("movies_ds/movies_metadata.csv")


# EDA

In [66]:
# Check rating range
print("\n--- Full Ratings Dataset ---")
print(f"Min rating: {ratings_df['rating'].min()}")
print(f"Max rating: {ratings_df['rating'].max()}")

# Unique keywords: some rows have empty lists, which is not null but still missing data
unique_keywords = keywords_df['id'].nunique()
print(f"\nUnique keywords: {unique_keywords}")


--- Full Ratings Dataset ---
Min rating: 0.5
Max rating: 5.0

Unique keywords: 45432


In [55]:

print("Credits: \n", credits_df.isnull().sum())
print("\nKeywords: \n", keywords_df.isnull().sum())
print("\nLinks: \n", links_df.isnull().sum())
print("\nRatings: \n", ratings_df .isnull().sum())
print("\nMetadata: \n", movies_metadata_df.isnull().sum())

Credits: 
 cast    0
crew    0
id      0
dtype: int64

Keywords: 
 id          0
keywords    0
dtype: int64

Links: 
 movieId      0
imdbId       0
tmdbId     219
dtype: int64

Ratings: 
 userId       0
movieId      0
rating       0
timestamp    0
dtype: int64

Metadata: 
 adult                        0
belongs_to_collection    40972
budget                       0
genres                       0
homepage                 37684
id                           0
imdb_id                     17
original_language           11
original_title               0
overview                   954
popularity                   5
poster_path                386
production_companies         3
production_countries         3
release_date                87
revenue                      6
runtime                    263
spoken_languages             6
status                      87
tagline                  25054
title                        6
video                        6
vote_average                 6
vote_count   

In [56]:
# Sample of movies_metadata_df null values in title
print("\nLinks: \n")
print(movies_metadata_df.loc[
    movies_metadata_df["title"].isnull(),
    ["original_title", "title"],
])


Links: 

                                 original_title title
19729                              Midnight Man   NaN
19730  [{'iso_639_1': 'en', 'name': 'English'}]   NaN
29502                          マルドゥック・スクランブル 排気   NaN
29503      [{'iso_639_1': 'ja', 'name': '日本語'}]   NaN
35586                          Avalanche Sharks   NaN
35587  [{'iso_639_1': 'en', 'name': 'English'}]   NaN


In [57]:
bad_title = movies_metadata_df[movies_metadata_df["title"].isnull()]
print(bad_title[["id", "adult", "original_title", "release_date", "vote_count"]])

ids_numeric = pd.to_numeric(movies_metadata_df["id"], errors="coerce")
print("Non-numeric ids:", movies_metadata_df.loc[ids_numeric.isnull(), "id"].tolist())

# 'adult' should only be True/False
print(movies_metadata_df["adult"].value_counts().head())

               id                                              adult  \
19729       82663                                              False   
19730  1997-08-20                                 - Written by Ørnås   
29502      122662                                              False   
29503  2012-09-29   Rune Balot goes to a casino connected to the ...   
35586      249260                                              False   
35587  2014-01-01   Avalanche Sharks tells the story of a bikini ...   

                                 original_title release_date  vote_count  
19729                              Midnight Man          NaN         NaN  
19730  [{'iso_639_1': 'en', 'name': 'English'}]            1         NaN  
29502                          マルドゥック・スクランブル 排気          NaN         NaN  
29503      [{'iso_639_1': 'ja', 'name': '日本語'}]           12         NaN  
35586                          Avalanche Sharks          NaN         NaN  
35587  [{'iso_639_1': 'en', 'name': 'English'

In [58]:
null_title_idx = movies_metadata_df.index[movies_metadata_df["title"].isnull()]
is_tail = pd.to_numeric(movies_metadata_df.loc[null_title_idx, "id"], errors="coerce").isnull()
tail_idx = null_title_idx[is_tail]
head_idx = null_title_idx[~is_tail]

assert len(head_idx) == len(tail_idx) == 3
assert (head_idx + 1 == tail_idx).all()

recovered_dates = movies_metadata_df.loc[tail_idx, "id"].values
assert pd.to_datetime(recovered_dates, format="%Y-%m-%d", errors="coerce").notnull().all()
movies_metadata_df.loc[head_idx, "release_date"] = recovered_dates

movies_metadata_df.loc[head_idx, "title"] = movies_metadata_df.loc[head_idx, "original_title"]

# Drop the malformed tail rows
movies_metadata_df = movies_metadata_df.drop(index=tail_idx)

# Verify
print("Null titles remaining:", movies_metadata_df["title"].isnull().sum())
print("Non-numeric ids remaining:", pd.to_numeric(movies_metadata_df["id"], errors="coerce").isnull().sum())
print(movies_metadata_df.loc[head_idx, ["id", "original_title", "title", "release_date"]])

Null titles remaining: 0
Non-numeric ids remaining: 0
           id    original_title             title release_date
19729   82663      Midnight Man      Midnight Man   1997-08-20
29502  122662  マルドゥック・スクランブル 排気  マルドゥック・スクランブル 排気   2012-09-29
35586  249260  Avalanche Sharks  Avalanche Sharks   2014-01-01


In [59]:
# movies_metadata sample of ids
print("\nSample of movies_metadata_df ids:")
print(movies_metadata_df.loc[head_idx, ["id", "imdb_id"]])

print("\nClean imdb_id in movies_metadata_df")
# Check how many start with 'tt'
sample_imdb = movies_metadata_df['imdb_id'].dropna().astype(str)
starts_with_tt = sample_imdb.str.startswith('tt').all()
print(f"All non-null imdb_id start with 'tt': {starts_with_tt}")

# Remove 'tt' prefix and convert remainder to nullable integer
movies_metadata_df['imdb_id_clean'] = movies_metadata_df['imdb_id'].astype(str).str.replace('tt', '', regex=False)
movies_metadata_df['imdb_id_clean'] = pd.to_numeric(movies_metadata_df['imdb_id_clean'], errors='coerce').astype('Int64')

print("Sample cleaned imdb_id values:")
print(movies_metadata_df[['imdb_id', 'imdb_id_clean']].head(3))

# Drop the old imdb_id column and rename the clean version
movies_metadata_df = movies_metadata_df.drop(columns=['imdb_id'])
movies_metadata_df = movies_metadata_df.rename(columns={'imdb_id_clean': 'imdb_id'})



Sample of movies_metadata_df ids:
           id    imdb_id
19729   82663  tt0113002
29502  122662  tt2423504
35586  249260  tt2622826

Clean imdb_id in movies_metadata_df
All non-null imdb_id start with 'tt': True
Sample cleaned imdb_id values:
     imdb_id  imdb_id_clean
0  tt0114709         114709
1  tt0113497         113497
2  tt0113228         113228
